# 18wC — Blind calibrated contract-probability release

The development-selected scale and smoothing parameters are applied to the frozen internal-holdout and June predictive quantiles. The release retains both uncalibrated and calibrated coherent contract probabilities.

No realised HKO value, contract outcome, realised residual, market price or market probability is loaded or exported. The identical pre-holdout fit is retained for June; no holdout-label refit is permitted.

**Revision v2.** The explicit schema-alignment concatenation suppresses the pandas all-NA FutureWarning; the resulting columns and values are unchanged.

**Revision v3.** Contract definitions are inherited from the outcome-blind 18wA probability panel. No 18s outcome or market file is opened.

In [1]:
from __future__ import annotations
import hashlib, json, platform, sys, warnings
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=Path.cwd().resolve()
if not (ROOT/'.git').exists(): raise RuntimeError(f'Run from repository root, not {ROOT}')
UTC=timezone.utc; STEP='18wC'; NQ=99; NC=11
VA=ROOT/'data/processed/18vA_residual_model_design_and_features'; VD=ROOT/'data/processed/18vD_selected_models_blind_predictions'; WA=ROOT/'data/processed/18wA_contract_probability_mapping'; WB=ROOT/'data/processed/18wB_development_probability_calibration'
QGRID=VA/'18vA_quantile_grid.csv'; VA_MAN=VA/'18vA_sha256_manifest.csv'; VA_SUM=VA/'18vA_summary.json'
BLIND_PRED=VD/'18vD_selected_candidate_blind_predictions.csv'; VD_MAN=VD/'18vD_sha256_manifest.csv'; VD_SUM=VD/'18vD_summary.json'
UNCAL=WA/'18wA_blind_uncalibrated_probability_panel.csv'; WA_MAN=WA/'18wA_sha256_manifest.csv'; WA_SUM=WA/'18wA_summary.json'
PARAM=WB/'18wB_selected_calibration_parameters.csv'; WB_MAN=WB/'18wB_sha256_manifest.csv'; WB_SUM=WB/'18wB_summary.json'
OUT=ROOT/'data/processed/18wC_blind_calibrated_probability_release'; REPORT=ROOT/'reports/18wC_blind_calibrated_probability_release'; OUT.mkdir(parents=True,exist_ok=True); REPORT.mkdir(parents=True,exist_ok=True)
for p in [QGRID,VA_MAN,VA_SUM,BLIND_PRED,VD_MAN,VD_SUM,UNCAL,WA_MAN,WA_SUM,PARAM,WB_MAN,WB_SUM]:
    if not p.is_file(): raise FileNotFoundError(p)

def sha(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for c in iter(lambda:f.read(1024*1024),b''): h.update(c)
    return h.hexdigest()
def verify(p):
    bad=[]
    for r in pd.read_csv(p).itertuples(index=False):
        q=ROOT/r.path
        if not q.is_file(): bad.append('MISSING '+r.path); continue
        if sha(q)!=r.sha256: bad.append('HASH '+r.path)
        if q.stat().st_size!=int(r.size_bytes): bad.append('SIZE '+r.path)
    if bad: raise AssertionError('\n'.join(bad))
def pbool(s,name):
    if pd.api.types.is_bool_dtype(s): return s.astype(bool)
    x=s.astype(str).str.strip().str.lower().map({'true':True,'false':False,'1':True,'0':False,'yes':True,'no':False})
    if x.isna().any(): raise ValueError(f'Cannot parse {name}')
    return x.astype(bool)
def order_book(g):
    z=g.copy(); z['_e']=z.event_type.map({'lower':0,'interior':1,'upper':2}); z['_l']=z.lower_bound_c.fillna(-1e9); z=z.sort_values(['_e','_l','upper_bound_c','market_id'],na_position='last').drop(columns=['_e','_l']).reset_index(drop=True); z['contract_order']=np.arange(len(z)); return z
def member(x,r):
    if r.event_type=='lower': return x<float(r.upper_bound_c)
    if r.event_type=='interior': return (x>=float(r.lower_bound_c))&(x<float(r.upper_bound_c))
    if r.event_type=='upper': return x>=float(r.lower_bound_c)
    raise ValueError(r.event_type)
def map_prob(q,forecast,book,scale,eta):
    q=np.asarray(q,float); center=q[49]; x=float(forecast)+center+float(scale)*(q-center); b=order_book(book); M=np.column_stack([member(x,r) for r in b.itertuples(index=False)])
    if not np.all(M.sum(1)==1): raise AssertionError('Particle mapping')
    counts=M.sum(0).astype(int); raw=counts/NQ; p=(1-float(eta))*raw+float(eta)/NC
    if counts.sum()!=NQ or not np.isclose(p.sum(),1,atol=1e-12): raise AssertionError('Book coherence')
    return b,counts,raw,p

for p in [VA_MAN,VD_MAN,WA_MAN,WB_MAN]: verify(p)
for name,p in [('18vA',VA_SUM),('18vD',VD_SUM),('18wA',WA_SUM),('18wB',WB_SUM)]:
    if json.loads(p.read_text()).get('verdict')!='PASS': raise AssertionError(f'{name} is not PASS')
qgrid=pd.read_csv(QGRID); blind=pd.read_csv(BLIND_PRED,low_memory=False); uncal=pd.read_csv(UNCAL,dtype={'market_id':str},low_memory=False); params=pd.read_csv(PARAM)
contracts=uncal.drop_duplicates(['event_date','market_id']).copy()
for df in [contracts,blind,uncal]: df['event_date']=pd.to_datetime(df.event_date,errors='raise')
blind['outcome_blind']=pbool(blind.outcome_blind,'outcome blind'); blind['refit_on_holdout_labels']=pbool(blind.refit_on_holdout_labels,'refit'); uncal['outcome_blind']=pbool(uncal.outcome_blind,'uncal outcome blind'); uncal['refit_on_holdout_labels']=pbool(uncal.refit_on_holdout_labels,'uncal refit')
rq=qgrid.residual_quantile_column.tolist(); books={d:order_book(g) for d,g in contracts.groupby('event_date')}
expected={'pooled_empirical_residual':(1.25,0.0),'gp_matern32_rule':(1.5,0.0),'catboost_quantile_pooled':(2.0,0.1)}
if len(params)!=3: raise AssertionError('Parameter rows')
for r in params.itertuples(index=False):
    if r.candidate_id not in expected or not np.isclose(r.scale_factor,expected[r.candidate_id][0]) or not np.isclose(r.uniform_smoothing,expected[r.candidate_id][1]): raise AssertionError('Unexpected calibration parameters')
forbidden={'hko_daily_max_c','residual_c','forecast_error_c','Y_event_int','Y_no_int','p_market','market_binary_brier','market_binary_log_score','current_label_available_utc'}
if forbidden.intersection(blind.columns) or forbidden.intersection(uncal.columns) or not blind.outcome_blind.all() or blind.refit_on_holdout_labels.any(): raise AssertionError('Blind boundary')
param_map=params.set_index('candidate_id')[['scale_factor','uniform_smoothing']].to_dict('index')
parts=[]
public=['event_date','market_id','condition_id','event_id','event_slug','market_slug','question','canonical_label','event_type','contract_event_type','lower_bound_c','upper_bound_c','bound_reference_c','selected_yes_token_id','no_token_id','sample_block','contract_order']
for r in blind.itertuples(index=False):
    par=param_map[r.candidate_id]; b,counts,raw,p=map_prob(np.array([getattr(r,c) for c in rq],float),r.forecast_daily_max_c,books[r.event_date],par['scale_factor'],par['uniform_smoothing']); z=b[[c for c in public if c in b.columns]].copy(); z['particle_count_scaled']=counts; z['p_particle_scaled']=raw; z['p_model_calibrated']=p
    for c,v in {'candidate_id':r.candidate_id,'model_family':r.model_family,'scope_type':r.scope_type,'scope_id':r.scope_id,'selection_roles':r.selection_roles,'complexity_rank':r.complexity_rank,'decision_rule':r.decision_rule,'decision_rule_order':r.decision_rule_order,'decision_cutoff_utc':r.decision_cutoff_utc,'evaluation_block':r.evaluation_block,'evaluation_stage':r.evaluation_stage,'forecast_daily_max_c':r.forecast_daily_max_c,'calibration_scale_factor':par['scale_factor'],'calibration_uniform_smoothing':par['uniform_smoothing'],'calibration_variant':'CALIBRATED','outcome_blind':True,'refit_on_holdout_labels':False,'fitted_parameter_source':r.fitted_parameter_source}.items(): z[c]=v
    parts.append(z)
cal=pd.concat(parts,ignore_index=True)
if len(cal)!=5247 or not cal.groupby(['candidate_id','event_date','decision_rule']).p_model_calibrated.sum().apply(lambda v:np.isclose(v,1,atol=1e-12)).all(): raise AssertionError('Calibrated blind panel')
if forbidden.intersection(cal.columns): raise AssertionError('Calibrated output outcomes')
# Standardise long release
u=uncal.copy(); u['probability_variant']='UNCALIBRATED'; u['p_model']=u.p_model_uncalibrated; u['calibration_scale_factor']=1.0; u['calibration_uniform_smoothing']=0.0
c=cal.copy(); c['probability_variant']='CALIBRATED'; c['p_model']=c.p_model_calibrated
common_cols=sorted(set(u.columns).union(c.columns))
for df in [u,c]:
    for col in common_cols:
        if col not in df.columns: df[col]=pd.NA
with warnings.catch_warnings():
    warnings.simplefilter('ignore',FutureWarning)
    release=pd.concat([u[common_cols],c[common_cols]],ignore_index=True)
if len(release)!=10494 or not release.groupby(['candidate_id','probability_variant','event_date','decision_rule']).p_model.sum().apply(lambda v:np.isclose(v,1,atol=1e-12)).all(): raise AssertionError('Combined blind release')
if forbidden.intersection(release.columns): raise AssertionError('Combined release outcomes')
# Blind book diagnostics without outcomes
bookrows=[]
for k,g in release.groupby(['candidate_id','probability_variant','evaluation_block','event_date','decision_rule'],sort=True):
    p=g.p_model.to_numpy(float); modal=g[np.isclose(g.p_model,p.max(),rtol=0,atol=1e-15)]
    entropy=float(-(p[p>0]*np.log(p[p>0])).sum())
    bookrows.append({'candidate_id':k[0],'probability_variant':k[1],'evaluation_block':k[2],'event_date':k[3],'decision_rule':k[4],'decision_rule_order':int(g.decision_rule_order.iloc[0]),'contract_rows':len(g),'probability_sum':float(p.sum()),'maximum_probability':float(p.max()),'modal_contract_count':len(modal),'modal_labels':'|'.join(modal.canonical_label.astype(str)),'entropy_nats':entropy,'calibration_scale_factor':float(g.calibration_scale_factor.iloc[0]),'calibration_uniform_smoothing':float(g.calibration_uniform_smoothing.iloc[0]),'outcome_blind':True})
bookinv=pd.DataFrame(bookrows)
if len(bookinv)!=954 or not np.isclose(bookinv.probability_sum,1,atol=1e-12).all(): raise AssertionError('Book inventory')
coverage=release.groupby(['candidate_id','probability_variant','evaluation_block']).agg(contract_rows=('market_id','size'),books=('event_date','size'),dates=('event_date','nunique')).reset_index(); coverage['books']=coverage.contract_rows//11
expected_counts={('INTERNAL_HOLDOUT',):40,('EXTERNAL_TEST',):119}
if not coverage[coverage.evaluation_block.eq('INTERNAL_HOLDOUT')].books.eq(40).all() or not coverage[coverage.evaluation_block.eq('EXTERNAL_TEST')].books.eq(119).all(): raise AssertionError('Evaluation coverage')
checks=pd.DataFrame([
{'check':'calibrated_probability_rows_5247','passed':len(cal)==5247,'detail':f'rows={len(cal)}','blocking':True},
{'check':'combined_probability_rows_10494','passed':len(release)==10494,'detail':f'rows={len(release)}','blocking':True},
{'check':'blind_book_inventory_rows_954','passed':len(bookinv)==954,'detail':f'rows={len(bookinv)}','blocking':True},
{'check':'holdout_books_per_candidate_variant_40','passed':coverage[coverage.evaluation_block.eq('INTERNAL_HOLDOUT')].books.eq(40).all(),'detail':'40','blocking':True},
{'check':'external_books_per_candidate_variant_119','passed':coverage[coverage.evaluation_block.eq('EXTERNAL_TEST')].books.eq(119).all(),'detail':'119','blocking':True},
{'check':'all_books_sum_to_one','passed':np.isclose(bookinv.probability_sum,1,atol=1e-12).all(),'detail':'coherent simplex','blocking':True},
{'check':'full_outcome_panel_not_loaded','passed':True,'detail':'18wC reads definitions from 18wA blind panel only','blocking':True},
{'check':'outcomes_absent','passed':not bool(forbidden.intersection(release.columns)),'detail':'outcome-blind release','blocking':True},
{'check':'no_holdout_label_refit','passed':not pbool(release.refit_on_holdout_labels,'release refit').any(),'detail':'identical pre-holdout fit','blocking':True},
{'check':'market_information_absent','passed':'p_market' not in release.columns,'detail':'market excluded','blocking':True},
])
if not checks.passed.all(): raise AssertionError(checks[~checks.passed].to_string(index=False))
issues=pd.DataFrame(columns=['issue_level','issue_code','candidate_id','evaluation_block','event_date','decision_rule','market_id','detail','blocking'])
outputs={'18wC_blind_calibrated_probability_panel.csv':cal,'18wC_blind_probability_release.csv':release,'18wC_blind_book_inventory.csv':bookinv,'18wC_blind_coverage_summary.csv':coverage,'18wC_selected_calibration_parameters.csv':params,'18wC_integrity_checks.csv':checks,'18wC_issues.csv':issues}
for name,df in outputs.items():
    z=df.copy()
    for col in z.columns:
        if 'date' in col.lower() and pd.api.types.is_datetime64_any_dtype(z[col]): z[col]=z[col].dt.strftime('%Y-%m-%d')
        if 'cutoff' in col.lower() or col.lower().endswith('_utc') or 'available' in col.lower() or 'initialisation' in col.lower(): z[col]=z[col].astype('string')
    z.to_csv(OUT/name,index=False)
protocol={'step':STEP,'generated_at_utc':datetime.now(UTC).isoformat(),'verdict':'PASS','probability_variants':['UNCALIBRATED','CALIBRATED'],'selected_parameters':params[['candidate_id','scale_factor','uniform_smoothing']].to_dict('records'),'blind_books_per_candidate':159,'internal_holdout_books_per_candidate':40,'external_books_per_candidate':119,'contract_definitions_source':'18wA blind uncalibrated probability panel','full_outcome_panel_loaded':False,'holdout_or_external_outcomes_loaded':False,'market_information_used':False,'refit_on_holdout_labels':False,'probability_bridge_retained':False,'evaluation_pending':True}
(OUT/'18wC_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
sources=pd.DataFrame([
{'input_role':'18vA_quantile_grid','path':str(QGRID.relative_to(ROOT)),'rows':len(qgrid),'sha256':sha(QGRID)},
{'input_role':'18vD_blind_predictions','path':str(BLIND_PRED.relative_to(ROOT)),'rows':len(blind),'sha256':sha(BLIND_PRED)},
{'input_role':'18wA_blind_uncalibrated_probabilities','path':str(UNCAL.relative_to(ROOT)),'rows':len(uncal),'sha256':sha(UNCAL)},
{'input_role':'18wB_selected_calibration_parameters','path':str(PARAM.relative_to(ROOT)),'rows':len(params),'sha256':sha(PARAM)},
]); sources.to_csv(OUT/'18wC_source_inventory.csv',index=False)
summary={'step':STEP,'generated_at_utc':datetime.now(UTC).isoformat(),'verdict':'PASS','selected_candidates':3,'probability_variants':2,'books_per_candidate_variant':159,'internal_holdout_books_per_candidate_variant':40,'external_books_per_candidate_variant':119,'calibrated_contract_probability_rows':5247,'combined_contract_probability_rows':10494,'blind_book_inventory_rows':954,'full_outcome_panel_loaded':False,'holdout_or_external_outcomes_loaded':False,'market_information_used':False,'refit_on_holdout_labels':False,'probability_bridge_retained':False,'evaluation_pending':True,'issue_rows':0,'integrity_checks_passed':int(checks.passed.sum()),'integrity_checks_total':len(checks)}
(OUT/'18wC_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
(OUT/'18wC_environment.json').write_text(json.dumps({'generated_at_utc':datetime.now(UTC).isoformat(),'python':sys.version,'platform':platform.platform(),'pandas':pd.__version__,'numpy':np.__version__,'revision':'v3'},indent=2),encoding='utf-8')
lines=['# 18wC blind calibrated contract-probability release','','**PASS**','','| Block | Candidate variants | Books per candidate and variant | Contract rows per candidate and variant |','|---|---:|---:|---:|','| Internal holdout | 6 | 40 | 440 |','| June external test | 6 | 119 | 1,309 |','','The combined release contains 10,494 coherent model-contract probabilities and no realised outcome or market field. June uses the identical pre-holdout fit.']
(REPORT/'18wC_blind_calibrated_probability_release_report.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
manifest=[]
for root in [OUT,REPORT]:
    for p in sorted(root.rglob('*')):
        if p.is_file() and p.name!='18wC_sha256_manifest.csv': manifest.append({'path':str(p.relative_to(ROOT)),'size_bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(manifest).to_csv(OUT/'18wC_sha256_manifest.csv',index=False)
print(json.dumps(summary,indent=2)); display(coverage); print('18wC blind calibrated probability release: PASS')

{
  "step": "18wC",
  "generated_at_utc": "2026-07-22T12:27:06.371588+00:00",
  "verdict": "PASS",
  "selected_candidates": 3,
  "probability_variants": 2,
  "books_per_candidate_variant": 159,
  "internal_holdout_books_per_candidate_variant": 40,
  "external_books_per_candidate_variant": 119,
  "calibrated_contract_probability_rows": 5247,
  "combined_contract_probability_rows": 10494,
  "blind_book_inventory_rows": 954,
  "full_outcome_panel_loaded": false,
  "holdout_or_external_outcomes_loaded": false,
  "market_information_used": false,
  "refit_on_holdout_labels": false,
  "probability_bridge_retained": false,
  "evaluation_pending": true,
  "issue_rows": 0,
  "integrity_checks_passed": 10,
  "integrity_checks_total": 10
}


,candidate_id,probability_variant,evaluation_block,contract_rows,books,dates
0,catboost_quantile_pooled,CALIBRATED,EXTERNAL_TEST,1309,119,30
1,catboost_quantile_pooled,CALIBRATED,INTERNAL_HOLDOUT,440,40,10
2,catboost_quantile_pooled,UNCALIBRATED,EXTERNAL_TEST,1309,119,30
3,catboost_quantile_pooled,UNCALIBRATED,INTERNAL_HOLDOUT,440,40,10
4,gp_matern32_rule,CALIBRATED,EXTERNAL_TEST,1309,119,30
5,gp_matern32_rule,CALIBRATED,INTERNAL_HOLDOUT,440,40,10
6,gp_matern32_rule,UNCALIBRATED,EXTERNAL_TEST,1309,119,30
7,gp_matern32_rule,UNCALIBRATED,INTERNAL_HOLDOUT,440,40,10
8,pooled_empirical_residual,CALIBRATED,EXTERNAL_TEST,1309,119,30
9,pooled_empirical_residual,CALIBRATED,INTERNAL_HOLDOUT,440,40,10


18wC blind calibrated probability release: PASS
